In [1]:
pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 43.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.metrics import f1_score, accuracy_score
from torch.optim.lr_scheduler import StepLR
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torchvision.transforms as T
from torchvision.transforms import InterpolationMode


import pytorch_lightning
from monai.transforms import (
    Activations,
)

from monai.data import Dataset, DataLoader
from pathlib import Path
import torch
import numpy as np
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.nn import BCEWithLogitsLoss
from torchmetrics import F1Score
from torch.optim.lr_scheduler import SequentialLR, LambdaLR, StepLR, SequentialLR
import ssl

from random import shuffle
import os
import random

from torchmetrics.classification import BinaryAUROC

import glob
import cv2
from skimage.filters import threshold_otsu
from scipy.stats import kurtosis, skew

import scipy
import scipy.ndimage as ndi

In [3]:
train_transforms = T.Compose([
    T.Resize(224, interpolation=InterpolationMode.BICUBIC),
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.GaussianBlur(kernel_size=(7, 13), sigma=(0.1, 1.0)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

val_transforms = T.Compose([
    T.Resize(224, interpolation=InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])


In [4]:
class Net(pytorch_lightning.LightningModule):
    def __init__(self, model, optimizer, scheduler, train_loader, val_loader):
        super().__init__()
        self._model = model
        self._optimizer = optimizer
        self._scheduler = scheduler
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_function = BCEWithLogitsLoss()
        self.metric = F1Score(task='binary')
        self.activation = Activations(sigmoid=True)


    def forward(self, x):
        return self._model(x)

    def train_dataloader(self):
        return self.train_loader

    def val_dataloader(self):
        return self.val_loader

    def configure_optimizers(self):
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}
    

    def training_step(self, batch, batch_idx):
        images, labels = batch
        labels = labels[:, None]
        output = self.forward(images)
        loss = self.loss_function(output, labels.float())
        self.log_dict({"training_loss": loss})
        return {"loss": loss}

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        labels = labels[:, None]
        output = self.forward(images)
        loss = self.loss_function(output, labels.float())
        metric = self.metric(self.activation(output), labels.float())
        self.log_dict({"f1":metric, "loss": loss})
        return {"loss": loss}

In [5]:
class ImageDataset(Dataset):
    def __init__(self, file_list, labels=None, transform=None):
        self.file_list = file_list
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.labels is not None:
            label = self.labels[idx]
            return img, label
        else:
            return img

In [6]:
class TestImageDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, os.path.basename(img_path) 

In [7]:
def load_test_df():
    """
    Loads and returns test dataframe in a proper form. 
    """
    base_dir = '/kaggle/input/ai-vs-human-generated-dataset'
    test_csv = '/kaggle/input/ai-vs-human-generated-dataset/test.csv'
    df_test = pd.read_csv(test_csv)
    df_test['id'] = df_test['id'].apply(lambda x: os.path.join(base_dir, x))
    return df_test

In [8]:
def get_train_valid_split():
    '''
    Creates train/valid split for cnn. 
    Return train_paths, valid_paths, train_labels, valid_labels.
    '''
    
    base_dir = '/kaggle/input/ai-vs-human-generated-dataset'
    train_csv = '/kaggle/input/ai-vs-human-generated-dataset/train.csv'
    
    
    df_train = pd.read_csv(train_csv)
    
    
    df_train = df_train[['file_name', 'label']]
    
    df_train['file_name'] = df_train['file_name'].apply(lambda x: os.path.join(base_dir, x))
    all_image_paths = df_train['file_name'].values
    all_labels = df_train['label'].values
    train_paths, val_paths, train_labels, val_labels = train_test_split(all_image_paths, 
                                                                        all_labels, test_size=0.05,        
                                                                        random_state=43,
                                                                        shuffle=False)
    return train_paths, val_paths, train_labels, val_labels

In [9]:
def create_dataloaders(train_paths, val_paths, train_labels, val_labels):
    """
    Returns train and valid dataloader.
    """
    batch_size = 32
    train_data = ImageDataset(train_paths, train_labels, transform=train_transforms)
    val_data   = ImageDataset(val_paths,   val_labels,   transform=val_transforms)
    train_loader = DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True,  num_workers=4)
    val_loader   = DataLoader(dataset=val_data,   batch_size=batch_size, shuffle=False, num_workers=4)
    return train_loader, val_loader

In [10]:
def create_model_optimizer_scheduler():
    """
    Returns model, optimizer and scheduler
    """

    model = models.convnext_base(weights="DEFAULT")
    
    for param in model.features.parameters():
        param.requires_grad = False
    
    for param in model.features[-2:].parameters(): 
        param.requires_grad = True
    
    model.classifier = nn.Sequential(
        nn.AdaptiveAvgPool2d((1, 1)),  
        nn.Flatten(),                  
        nn.BatchNorm1d(1024),          
        nn.Linear(1024, 512),          
        nn.ReLU(),                     
        nn.Dropout(0.4),               
        nn.Linear(512, 1)              
    )
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    optimizer = torch.optim.AdamW([
        {'params': model.features[-2:].parameters(), 'lr': 1e-5},
        {'params': model.classifier.parameters(), 'lr': 1e-4}     
    ])
    
    scheduler = StepLR(optimizer, step_size=5, gamma=0.7)

    return model, optimizer, scheduler

In [11]:
def train_model(model, optimizer, scheduler, train_loader, val_loader):
    """
    Trains model
    """
    net = Net(model, optimizer, scheduler, train_loader, val_loader)
    trainer = pytorch_lightning.Trainer(
            devices=[0],
            max_epochs=10,
            enable_checkpointing=True,
            num_sanity_val_steps=1,
            log_every_n_steps=16,
            callbacks=[ModelCheckpoint('models/', '{f1:.2f}_{epoch}', monitor='f1', mode='max')]
        )
    trainer.fit(net)
    return net

In [12]:
def predict(model, df_test):
    """
    Predicts on test dataframe.
    Returns dataframe with id, label and logits.
    """
    model.cuda().eval()
    test_dataset = TestImageDataset(df_test['id'].values, transform=val_transforms)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)
    predictions = []
    image_names = []
    
    with torch.no_grad():
        for data, names in tqdm(test_loader, desc="Predicting"):
            data = data.to('cuda')
            outputs = model(data)
            
            predictions.extend(outputs.cpu().numpy())
            image_names.extend([f"test_data_v2/{name}" for name in names])

    predictions = np.array(predictions)[:, 0]
    submission_df = pd.DataFrame({
        'id': image_names,
        'label': np.uint8((predictions > 0).tolist()),
        'logits': predictions
    })
    return submission_df

In [13]:
train_paths, val_paths, train_labels, val_labels = get_train_valid_split()
train_loader, val_loader = create_dataloaders(train_paths, val_paths, train_labels, val_labels)
model, optimizer, scheduler = create_model_optimizer_scheduler()
net = train_model(model, optimizer, scheduler, train_loader, val_loader)
df_test = load_test_df()
cnn_prediction_df = predict(net, df_test)

Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth
100%|██████████| 338M/338M [00:01<00:00, 200MB/s]


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: 100%|██████████| 174/174 [01:19<00:00,  2.19it/s]


In [14]:
from sklearn.cluster import KMeans


def get_stats(file):
    """
    Gets statistics from an image. 
    """
    
    img = cv2.imread(str(file))
    
    h, w = img.shape[:2]
    hm = max(h, w)
    wm = min(h, w)
    ratio = hm / wm
    size = os.stat(file).st_size / 1024 ** 2
    smooth = np.average(np.absolute(ndi.filters.laplace(img.astype(float) / 255.0)))
    median_filter = ndi.median_filter(img.astype(float), size=3)
    noise = np.mean(np.absolute(median_filter - img))
    base = [hm, wm, ratio, size, smooth, noise]
    if len(img.shape) == 3:
        for i in range(3):
            channel = img[:,:,i]
            mean = channel.mean()
            std = channel.std()
            q1 = np.quantile(channel, 0.25)
            q3 = np.quantile(channel, 0.75)
            m = channel.min()
            M = channel.max()
            th = threshold_otsu(channel)
            k = kurtosis(channel, axis=(0,1))
            s = skew(channel, axis=(0,1))
            temp = [mean, std, q1, q3, q3-q1, m, M, M-m, th, s]
            base += temp
    else:
        channel = img
        mean = channel.mean()
        std = channel.std()
        q1 = np.quantile(channel, 0.25)
        q3 = np.quantile(channel, 0.75)
        m = channel.min()
        M = channel.max()
        th = threshold_otsu(channel)
        k = kurtosis(channel, axis=(0,1))
        s = skew(channel, axis=(0,1))
        temp = [mean, std, q1, q3, q3-q1, m, M, M-m, th, s] * 3
        base += temp

    return base

In [15]:
def create_stats(files):
    '''
    Creates statistics for selected images. 
    '''
    stats = []
    for f in tqdm(files):
        stats += [get_stats(f)]
    stats = np.array(stats)
    return stats
    

def fit_and_predict_kmeans(stats, files):
    """
    Trains kmeans model and returns predictions. 
    """
    kmeans = KMeans(2, random_state=2)
    kmeans.fit(stats)
    preds_kmeans = kmeans.predict(stats)
    file_idxs = ['/'.join(f.split('/')[-2:]) for f in files]
    df_kmeans = pd.DataFrame({'id': file_idxs, 'label': preds_kmeans})
    return df_kmeans


def set_label(df_kmeans, cnn_prediction_df):
    """
    Reverses labels if necessary.
    """
    label = df_kmeans['label'].values
    label_kmeans = df_kmeans.sort_values(by='id')['label'].values
    label_cnn = cnn_prediction_df.sort_values(by='id')['label'].values
    acc_no_change = (label_kmeans == label_cnn).mean()
    acc_change = (label_kmeans != label_cnn).mean()
    if acc_no_change < acc_change:
        label = np.where(label==0, 1, 0)
        df_kmeans['label'] = label
    return df_kmeans


def insert_confident_cnn_predictions(df_kmeans, cnn_prediction_df):
    """
    Inserts confident top 1500 cnn predictions as 1 and tail 50 as 0. 
    """
    ones = cnn_prediction_df.sort_values(by='logits').tail(1500)['id'].values
    zeros = cnn_prediction_df.sort_values(by='logits').head(50)['id'].values
    df_kmeans.loc[df_kmeans['id'].isin(ones), 'label'] = 1
    df_kmeans.loc[df_kmeans['id'].isin(zeros), 'label'] = 0
    return df_kmeans

In [16]:
files = glob.glob('/kaggle/input/ai-vs-human-generated-dataset/test_data_v2/**')
stats = create_stats(files)

  0%|          | 0/5540 [00:00<?, ?it/s]<ipython-input-14-391abff9c8f2>:16: DeprecationWarning: Please import `laplace` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` namespace is deprecated and will be removed in SciPy 2.0.0.
  smooth = np.average(np.absolute(ndi.filters.laplace(img.astype(float) / 255.0)))
100%|██████████| 5540/5540 [1:41:06<00:00,  1.10s/it]


In [17]:
df_kmeans = fit_and_predict_kmeans(stats, files)
df_kmeans = set_label(df_kmeans, cnn_prediction_df)
df_kmeans = insert_confident_cnn_predictions(df_kmeans, cnn_prediction_df)
df_kmeans.to_csv('submission.csv', index=False)

/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
